In [1]:
import pyspark
from pyspark.sql.functions import substring, col, when, lit, regexp_extract,  regexp_replace

In [2]:
spark = pyspark.sql.SparkSession.builder.appName("silver").config("spark.driver.memory", "12g").getOrCreate()

# Print the SparkSession details to verify
print(spark)

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
25/04/04 19:26:22 WARN Utils: Your hostname, Grey resolves to a loopback address: 127.0.1.1; using 172.24.242.6 instead (on interface eth0)
25/04/04 19:26:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/04 19:26:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/04 19:26:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/04 19:26:24 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
all_images_no_labels = spark.read.parquet("./bronze/idimage.parquet")
non_fraud_images = spark.read.parquet("./bronze/idmeta.parquet")
#fraud_images = spark.read.parquet("./bronze/idlabel.parquet")
fraud_meta_supplement = spark.read.parquet("./bronze/idfraudmeta.parquet")

In [4]:
non_fraud_images = non_fraud_images.withColumnRenamed('name', 'person_name')
fraud_meta_supplement = fraud_meta_supplement.withColumnRenamed('name', 'person_name')

In [5]:
#print(f'total images: {all_images_no_labels.count()}')
#print(f'total non fraud images: {non_fraud_images.count()}')
#print(f'total fraud images: {fraud_images.count()}')

In [6]:
spark.sql("select * from {all_images_no_labels}", all_images_no_labels=all_images_no_labels).show(truncate=True)

+--------------------+--------------------+--------------------+--------------------+
|                name|           imageData|    unencoded_base64|          img_tensor|
+--------------------+--------------------+--------------------+--------------------+
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 4

In [7]:
all_images_labeled = all_images_no_labels.withColumn(
        'prelabeled_fraud',
        when(col('name').contains('fake'), lit(1)).otherwise(lit(0)))

In [8]:
all_images_labeled.show()

+--------------------+--------------------+--------------------+--------------------+----------------+
|                name|           imageData|    unencoded_base64|          img_tensor|prelabeled_fraud|
+--------------------+--------------------+--------------------+--------------------+----------------+
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|               0|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|               0|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|               0|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|               0|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|               0|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1.0, 1.0,...|               0|
|generated.photos_...|iVBORw0KGgoAAAANS...|[89 50 4E 47 0D 0...|[[[1.0, 1

In [9]:
all_images_labeled.write.mode("overwrite").parquet("./silver/all_images_labeled.parquet")

In [10]:
all_images_labeled_with_meta = spark.read.parquet("./silver/all_images_labeled.parquet")

below gets all real images with associated meta, should be 799 images. 

In [11]:
real_image_meta_labeled = spark.sql('select * from {all_images_labeled_with_meta} as all_images inner join {non_fraud_images} as real_meta on real_meta.id=all_images.name', all_images_labeled_with_meta=all_images_labeled_with_meta, non_fraud_images=non_fraud_images)

below gets all the modified fake images, type 5 or 6, by joing a partial of the names together. 

In [12]:
substring = '_fake_'

In [13]:
typ56fraud = all_images_labeled.withColumn('name_fixed', regexp_extract(col('name'), f"^(.*{substring})", 1))


In [14]:
typ56fraud_2 = typ56fraud.withColumn("name_fixed", regexp_replace('name_fixed', '_fake_', ''))

In [15]:
typ56fraud_final = typ56fraud_2.filter(typ56fraud_2.name_fixed != 'generated')

In [16]:
typ56fraud_final = typ56fraud_final.withColumnRenamed('name', 'name_old')
typ56fraud_final = typ56fraud_final.withColumnRenamed('name_fixed', 'name')
typ56fraud_final = typ56fraud_final.drop('name_old')

In [17]:
fraud_image_meta_labeled = spark.sql('select * from {typ56fraud_final} as all_images inner join {non_fraud_images} as fake_meta on fake_meta.id=all_images.name and all_images.prelabeled_fraud=1', non_fraud_images=non_fraud_images, typ56fraud_final=typ56fraud_final)


In [18]:
fraud_image_meta_labeled.count()

99

join the purely artifical images to meta

In [19]:
fake_meta_supplement_jpg = fraud_meta_supplement.withColumn("id", regexp_replace('id', '.jpg', ''))

In [20]:
fake_image_meta_labeled = fake_image_meta_labeled = spark.sql('select * from {all_images_labeled_with_meta} as all_images inner join {fake_meta_supplement_jpg} as fake_meta on fake_meta.id=all_images.name', all_images_labeled_with_meta=all_images_labeled_with_meta, fake_meta_supplement_jpg=fake_meta_supplement_jpg)

In [21]:
#labeled_training_set.write.mode("overwrite").parquet("./silver/labeled_training_set.parquet")

In [22]:
union1 = real_image_meta_labeled.unionByName(fake_image_meta_labeled)
union2 =  union1.unionByName(fraud_image_meta_labeled)

In [23]:
union2.count()

998

In [24]:
union2.repartition(numPartitions=998)

DataFrame[name: string, imageData: string, unencoded_base64: binary, img_tensor: array<array<array<float>>>, prelabeled_fraud: int, id: string, person_name: string, address: string, birthday: string, gender: string, ethnicity: string, class: string, issue_date: string, expire_date: string, eye_color: string, hair_color: string, is_donor: string, is_veteran: string, license_number: string, height_inches: int, weight_lbs: string]

In [25]:
union2 = union2.drop('imageData')
union2 = union2.drop('unencoded_base64')


In [26]:
union2.write.mode("overwrite").parquet("./silver/labeled_training_set.parquet")